In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import matplotlib.pyplot as plt
import os
import math
import ipywidgets as widgets
from IPython.display import display

In [ ]:
UPSCALE_FACTOR = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR = "results"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = "netG_final.pth"

In [ ]:
class UpsampleBLock(nn.Module):

    def __init__(self, in_channels, up_scale):
        super(UpsampleBLock, self).__init__()

        self.conv = nn.Conv2d(in_channels, in_channels * up_scale ** 2, 3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(up_scale)
        self.prelu = nn.PReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.pixel_shuffle(x)
        x = self.prelu(x)
        return x


class ResidualBlock(nn.Module):

    def __init__(self, channels):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.prelu = nn.PReLU()

        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):

        residual = self.conv1(x)
        residual = self.bn1(residual)
        residual = self.prelu(residual)

        residual = self.conv2(residual)
        residual = self.bn2(residual)

        return x + residual

In [ ]:
class Generator(nn.Module):

    def __init__(self, scale_factor):

        super(Generator, self).__init__()

        upsample_block_num = int(math.log(scale_factor, 2))

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 9, padding=4),
            nn.PReLU()
        )

        self.block2 = ResidualBlock(64)
        self.block3 = ResidualBlock(64)
        self.block4 = ResidualBlock(64)
        self.block5 = ResidualBlock(64)
        self.block6 = ResidualBlock(64)

        self.block7 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64)
        )

        block8 = [UpsampleBLock(64, 2) for _ in range(upsample_block_num)]
        block8.append(nn.Conv2d(64, 3, 9, padding=4))

        self.block8 = nn.Sequential(*block8)

    def forward(self, x):

        block1 = self.block1(x)

        block2 = self.block2(block1)
        block3 = self.block3(block2)
        block4 = self.block4(block3)
        block5 = self.block5(block4)
        block6 = self.block6(block5)

        block7 = self.block7(block6)

        block8 = self.block8(block1 + block7)

        return (torch.tanh(block8) + 1) / 2

In [ ]:
netG = Generator(UPSCALE_FACTOR).to(DEVICE)

state = torch.load(MODEL_PATH, map_location=DEVICE)
netG.load_state_dict(state)

netG.eval()

print("Generator loaded successfully!")

In [ ]:
upload = widgets.FileUpload(
    accept='image/*',
    multiple=False
)

display(upload)

for filename in upload.value:

    content = upload.value[filename]['content']

    with open(filename, "wb") as f:
        f.write(content)

    lr_image = Image.open(filename).convert("RGB")

    transform = transforms.ToTensor()
    lr_tensor = transform(lr_image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        sr_tensor = netG(lr_tensor)

    save_path = os.path.join(SAVE_DIR, "SR_" + filename)

    save_image(sr_tensor, save_path)

    print("Saved:", save_path)

    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.imshow(lr_image)
    plt.title("Low Resolution")
    plt.axis("off")

    plt.subplot(1,2,2)
    sr_image = transforms.ToPILImage()(sr_tensor.squeeze().cpu())
    plt.imshow(sr_image)
    plt.title("Super Resolution")
    plt.axis("off")

    plt.show()

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import os
import ipywidgets as widgets
from IPython.display import display

In [ ]:
class UpsampleBLock(nn.Module):

    def __init__(self, in_channels, up_scale):
        super(UpsampleBLock, self).__init__()

        self.conv = nn.Conv2d(in_channels, in_channels * (up_scale ** 2), 3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(up_scale)
        self.prelu = nn.PReLU()

    def forward(self, x):

        x = self.conv(x)
        x = self.pixel_shuffle(x)

        return self.prelu(x)


class ResidualBlock(nn.Module):

    def __init__(self, channels):

        super(ResidualBlock, self).__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.PReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x):

        return x + self.block(x)


class SuperResolutionCNN(nn.Module):

    def __init__(self):

        super(SuperResolutionCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.prelu = nn.PReLU()

        res_layers = [ResidualBlock(64) for _ in range(6)]
        self.res_blocks = nn.Sequential(*res_layers)

        self.upsample = nn.Sequential(
            UpsampleBLock(64, 2),
            UpsampleBLock(64, 2)
        )

        self.conv_final = nn.Conv2d(64, 3, 3, padding=1)

    def forward(self, x):

        f1 = self.prelu(self.conv1(x))

        res = self.res_blocks(f1)

        up = self.upsample(f1 + res)

        return self.conv_final(up)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "cnn_final.pth"   # model file in your repo

model = SuperResolutionCNN().to(device)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

model.eval()

print("✅ Model loaded successfully!")

In [ ]:
upload = widgets.FileUpload(
    accept='image/*',
    multiple=False
)

display(upload)

In [ ]:
for filename in upload.value:

    content = upload.value[filename]['content']

    with open(filename, "wb") as f:
        f.write(content)

    img = Image.open(filename).convert("RGB")

    transform = transforms.Compose([
        transforms.Resize((22,22)),   # LR input size
        transforms.ToTensor()
    ])

    lr_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        sr_tensor = model(lr_tensor)
        sr_tensor = torch.clamp(sr_tensor,0,1)

    lr_display = lr_tensor.squeeze(0).cpu().permute(1,2,0)
    sr_display = sr_tensor.squeeze(0).cpu().permute(1,2,0)

    plt.figure(figsize=(12,6))

    plt.subplot(1,2,1)
    plt.imshow(lr_display)
    plt.title("Low Resolution (22x22)")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(sr_display)
    plt.title("Super Resolution (88x88)")
    plt.axis("off")

    plt.show()

    os.makedirs("results",exist_ok=True)

    output_img = transforms.ToPILImage()(sr_tensor.squeeze(0).cpu())

    save_path = f"results/upscaled_{filename}"

    output_img.save(save_path)

    print("Saved:", save_path)